# Implementation of Random Forest

In [25]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import warnings
warnings.filterwarnings("ignore")
%matplotlib inline

In [26]:
df=pd.read_csv('Travel.csv')
df.head()

,CustomerID,ProdTaken,Age,TypeofContact,CityTier,DurationOfPitch,Occupation,Gender,NumberOfPersonVisiting,NumberOfFollowups,ProductPitched,PreferredPropertyStar,MaritalStatus,NumberOfTrips,Passport,PitchSatisfactionScore,OwnCar,NumberOfChildrenVisiting,Designation,MonthlyIncome
0,200000,1,41.0,Self Enquiry,3,6.0,Salaried,Female,3,3.0,Deluxe,3.0,Single,1.0,1,2,1,0.0,Manager,20993.0
1,200001,0,49.0,Company Invited,1,14.0,Salaried,Male,3,4.0,Deluxe,4.0,Divorced,2.0,0,3,1,2.0,Manager,20130.0
2,200002,1,37.0,Self Enquiry,1,8.0,Free Lancer,Male,3,4.0,Basic,3.0,Single,7.0,1,3,0,0.0,Executive,17090.0
3,200003,0,33.0,Company Invited,1,9.0,Salaried,Female,2,3.0,Basic,3.0,Divorced,2.0,1,5,1,1.0,Executive,17909.0
4,200004,0,NaN,Self Enquiry,1,8.0,Small Business,Male,2,3.0,Basic,4.0,Divorced,1.0,0,5,1,0.0,Executive,18468.0


In [27]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 4888 entries, 0 to 4887
Data columns (total 20 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   CustomerID                4888 non-null   int64  
 1   ProdTaken                 4888 non-null   int64  
 2   Age                       4662 non-null   float64
 3   TypeofContact             4863 non-null   str    
 4   CityTier                  4888 non-null   int64  
 5   DurationOfPitch           4637 non-null   float64
 6   Occupation                4888 non-null   str    
 7   Gender                    4888 non-null   str    
 8   NumberOfPersonVisiting    4888 non-null   int64  
 9   NumberOfFollowups         4843 non-null   float64
 10  ProductPitched            4888 non-null   str    
 11  PreferredPropertyStar     4862 non-null   float64
 12  MaritalStatus             4888 non-null   str    
 13  NumberOfTrips             4748 non-null   float64
 14  Passport           

### Handling Misiing Values

In [28]:
df.isnull().sum()

CustomerID                    0
ProdTaken                     0
Age                         226
TypeofContact                25
CityTier                      0
DurationOfPitch             251
Occupation                    0
Gender                        0
NumberOfPersonVisiting        0
NumberOfFollowups            45
ProductPitched                0
PreferredPropertyStar        26
MaritalStatus                 0
NumberOfTrips               140
Passport                      0
PitchSatisfactionScore        0
OwnCar                        0
NumberOfChildrenVisiting     66
Designation                   0
MonthlyIncome               233
dtype: int64

In [29]:
features_with_na=[feature for feature in df.columns if df[feature].isnull().any()]
for feature in features_with_na:
    print(feature,np.round(df[feature].isnull().mean()*100,5))

Age 4.62357
TypeofContact 0.51146
DurationOfPitch 5.13502
NumberOfFollowups 0.92062
PreferredPropertyStar 0.53191
NumberOfTrips 2.86416
NumberOfChildrenVisiting 1.35025
MonthlyIncome 4.76678


In [30]:
df['Age']=df['Age'].fillna(df['Age'].median())
df['TypeofContact']=df['TypeofContact'].fillna(df['TypeofContact'].mode()[0])
df['DurationOfPitch']=df['DurationOfPitch'].fillna(df['DurationOfPitch'].median())
df['NumberOfFollowups']=df['NumberOfFollowups'].fillna(df['NumberOfFollowups'].mode()[0])
df['PreferredPropertyStar']=df['PreferredPropertyStar'].fillna(df['PreferredPropertyStar'].mode()[0])
df['NumberOfTrips']=df['NumberOfTrips'].fillna(df['NumberOfTrips'].median())
df['NumberOfChildrenVisiting']=df['NumberOfChildrenVisiting'].fillna(df['NumberOfChildrenVisiting'].mode()[0])
df['MonthlyIncome']=df['MonthlyIncome'].fillna(df['MonthlyIncome'].median())







In [31]:
df.drop('CustomerID',inplace=True,axis=1)

In [32]:
df.isnull().sum()

ProdTaken                   0
Age                         0
TypeofContact               0
CityTier                    0
DurationOfPitch             0
Occupation                  0
Gender                      0
NumberOfPersonVisiting      0
NumberOfFollowups           0
ProductPitched              0
PreferredPropertyStar       0
MaritalStatus               0
NumberOfTrips               0
Passport                    0
PitchSatisfactionScore      0
OwnCar                      0
NumberOfChildrenVisiting    0
Designation                 0
MonthlyIncome               0
dtype: int64

### Check for categories 

In [33]:
df['ProdTaken'].value_counts()

ProdTaken
0    3968
1     920
Name: count, dtype: int64

In [34]:
df['TypeofContact'].value_counts()

TypeofContact
Self Enquiry       3469
Company Invited    1419
Name: count, dtype: int64

In [35]:
df['Gender'].value_counts()

Gender
Male       2916
Female     1817
Fe Male     155
Name: count, dtype: int64

In [36]:
df['MaritalStatus'].value_counts()

MaritalStatus
Married      2340
Divorced      950
Single        916
Unmarried     682
Name: count, dtype: int64

In [37]:
df['Gender']=df['Gender'].replace('Fe Male','Female')
df['MaritalStatus']=df['MaritalStatus'].replace('Unmarried','Single')

In [38]:
df.head()

,ProdTaken,Age,TypeofContact,CityTier,DurationOfPitch,Occupation,Gender,NumberOfPersonVisiting,NumberOfFollowups,ProductPitched,PreferredPropertyStar,MaritalStatus,NumberOfTrips,Passport,PitchSatisfactionScore,OwnCar,NumberOfChildrenVisiting,Designation,MonthlyIncome
0,1,41.0,Self Enquiry,3,6.0,Salaried,Female,3,3.0,Deluxe,3.0,Single,1.0,1,2,1,0.0,Manager,20993.0
1,0,49.0,Company Invited,1,14.0,Salaried,Male,3,4.0,Deluxe,4.0,Divorced,2.0,0,3,1,2.0,Manager,20130.0
2,1,37.0,Self Enquiry,1,8.0,Free Lancer,Male,3,4.0,Basic,3.0,Single,7.0,1,3,0,0.0,Executive,17090.0
3,0,33.0,Company Invited,1,9.0,Salaried,Female,2,3.0,Basic,3.0,Divorced,2.0,1,5,1,1.0,Executive,17909.0
4,0,36.0,Self Enquiry,1,8.0,Small Business,Male,2,3.0,Basic,4.0,Divorced,1.0,0,5,1,0.0,Executive,18468.0


### Feature Engineering

In [22]:
df['TotalVisiting']=df['NumberOfPersonVisiting']+df['NumberOfChildrenVisiting']
df.drop(columns=['NumberOfPersonVisiting','NumberOfChildrenVisiting'],inplace=True)


In [23]:
df.head()

,ProdTaken,Age,TypeofContact,CityTier,DurationOfPitch,Occupation,Gender,NumberOfFollowups,ProductPitched,PreferredPropertyStar,MaritalStatus,NumberOfTrips,Passport,PitchSatisfactionScore,OwnCar,Designation,MonthlyIncome,TotalVisiting
0,1,41.0,Self Enquiry,3,6.0,Salaried,Female,3.0,Deluxe,3.0,Single,1.0,1,2,1,Manager,20993.0,3.0
1,0,49.0,Company Invited,1,14.0,Salaried,Male,4.0,Deluxe,4.0,Divorced,2.0,0,3,1,Manager,20130.0,5.0
2,1,37.0,Self Enquiry,1,8.0,Free Lancer,Male,4.0,Basic,3.0,Single,7.0,1,3,0,Executive,17090.0,3.0
3,0,33.0,Company Invited,1,9.0,Salaried,Female,3.0,Basic,3.0,Divorced,2.0,1,5,1,Executive,17909.0,3.0
4,0,36.0,Self Enquiry,1,8.0,Small Business,Male,3.0,Basic,4.0,Divorced,1.0,0,5,1,Executive,18468.0,2.0


In [41]:
#encoding categorical features
cat_features=X.select_dtypes(include="object").columns
num_features=X.select_dtypes(exclude="object").columns
from sklearn.preprocessing import OneHotEncoder , StandardScaler
from sklearn.compose import ColumnTransformer
numeric_transformer=StandardScaler()
cat_transformer=OneHotEncoder(drop='first')
preprocessor=ColumnTransformer(
    [
        ("OneHotEncoder",cat_transformer,cat_features),
        ("StandardScaler",numeric_transformer,num_features)
    ]
)

In [42]:
X_train=preprocessor.fit_transform(X_train)
X_test=preprocessor.transform(X_test)

### Model Training

In [39]:
X=df.drop(['ProdTaken'],axis=1)
Y=df['ProdTaken']

In [40]:
from sklearn.model_selection import train_test_split
X_train,X_test,Y_train,Y_test=train_test_split(X,Y,test_size=0.30,random_state=10)

In [43]:
from sklearn.ensemble import  RandomForestClassifier 
from sklearn.metrics import accuracy_score,classification_report,confusion_matrix,precision_score,recall_score,f1_score,roc_auc_score


In [50]:
models={
    "randomforest":RandomForestClassifier(n_estimators= 200, min_samples_split=2, max_features=8, max_depth=None)
}
for i in range(len(list(models))):
    model=list(models.values())[i]
    model.fit(X_train,Y_train)
    y_train_pred=model.predict(X_train)
    y_test_pred=model.predict(X_test)
    #training dataa
    model_train_accuracy=accuracy_score(Y_train,y_train_pred)    
    model_train_f1=f1_score(Y_train,y_train_pred)
    model_train_precision=precision_score(Y_train,y_train_pred)
    model_train_recall=recall_score(Y_train,y_train_pred)
    model_train_rocauc_score=roc_auc_score(Y_train,y_train_pred)
    #testing data
    model_test_accuracy=accuracy_score(Y_test,y_test_pred)
    model_test_f1=f1_score(Y_test,y_test_pred)
    model_test_precision=precision_score(Y_test,y_test_pred)
    model_test_recall=recall_score(Y_test,y_test_pred)
    model_test_rocauc_score=roc_auc_score(Y_test,y_test_pred)
    print(list(models.keys())[i])
    print('Model performance for Training Data Set')
    print(f"Model training accuracy:{model_train_accuracy}")
    print(f"Model training f1-score:{model_train_f1}")
    print(f"Model training precision:{ model_train_precision}")
    print(f"Model training recall:{model_train_recall}")
    print(f"Model training rocauc score:{model_train_rocauc_score}")


    print('Model performance for Test Data Set')
    print(f"Model test accuracy:{model_test_accuracy}")
    print(f"Model test f1-score:{model_test_f1}")
    print(f"Model test precision:{ model_test_precision}")
    print(f"Model test recall:{model_test_recall}")
    print(f"Model test rocauc score:{model_test_rocauc_score}")

    

randomforest
Model performance for Training Data Set
Model training accuracy:1.0
Model training f1-score:1.0
Model training precision:1.0
Model training recall:1.0
Model training rocauc score:1.0
Model performance for Test Data Set
Model test accuracy:0.9386503067484663
Model test f1-score:0.7935779816513762
Model test precision:0.9105263157894737
Model test recall:0.7032520325203252
Model test rocauc score:0.8446645092986557


### Hyperparameter Training

In [47]:
params={
    "n_estimators":[100,200,500,1000],
    "max_depth":[5,8,15,None,10],
    "max_features":[5,7,"auto",8],
    "min_samples_split":[2,8,15,20]
}

In [48]:
randomcv_model=[
    ("RF",RandomForestClassifier(),params),
]

In [49]:
from sklearn.model_selection import RandomizedSearchCV
model_param={}
for name,model,params in randomcv_model:
    random=RandomizedSearchCV(estimator=model,param_distributions=params,n_iter=100,cv=3,verbose=2,n_jobs=-1)
    random.fit(X_train,Y_train)
    model_param[name]=random.best_params_
for model_name in model_param:
    print(f"Beat params for model {model_name}")
    print(model_param[model_name])
    


Fitting 3 folds for each of 100 candidates, totalling 300 fits
Beat params for model RF
{'n_estimators': 200, 'min_samples_split': 2, 'max_features': 8, 'max_depth': None}
